# 02 – Advanced Retrieval with LlamaIndex

In this notebook we go beyond the quickstart to explore:
1. Configuring **`Settings`** (the v0.10+ replacement for `ServiceContext`) to control the LLM, embedding model, chunk size, and chunk overlap.
2. Building a `VectorStoreIndex` with custom chunking parameters.
3. Adding an **`LLMRerank`** post-processor to improve the quality of retrieved context before generation.

> **Requirements**: Make sure `data/paul_graham_essay.txt` exists (run notebook 01 or the `postBuild` script first).

## 0. Install dependencies

Uncomment the cell below if you are running outside of Binder and have not yet installed the requirements.

In [1]:
# %pip install llama-index openai python-dotenv ipywidgets -q

## 1. Set your OpenAI API Key

In [2]:
import os
from getpass import getpass

# Enter your OpenAI API key when prompted.
# The key is stored only in memory for this session and is never saved to disk.
openai_api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

## 2. (Optional) Download the essay

Skip this cell if `data/paul_graham_essay.txt` already exists.

In [3]:
import urllib.request
import os

os.makedirs("data", exist_ok=True)

essay_path = "data/paul_graham_essay.txt"
if not os.path.exists(essay_path):
    url = (
        "https://raw.githubusercontent.com/run-llama/llama_index/"
        "main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt"
    )
    urllib.request.urlretrieve(url, essay_path)
    print(f"Downloaded essay to {essay_path}")
else:
    print(f"Essay already exists at {essay_path}")

Essay already exists at data/paul_graham_essay.txt


## 3. Configure `Settings` (LlamaIndex v0.10+)

In LlamaIndex v0.10+, `ServiceContext` has been replaced by a global `Settings` object.
You can customise it once at the top of your notebook and every component will pick up those defaults.

| Setting | Description | Default |
|---|---|---|
| `llm` | LLM used for synthesis | `gpt-3.5-turbo` |
| `embed_model` | Embedding model | `text-embedding-ada-002` |
| `chunk_size` | Number of tokens per chunk | 1024 |
| `chunk_overlap` | Token overlap between consecutive chunks | 20 |

In [4]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# --- LLM ------------------------------------------------------------------
Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)

# --- Embedding model ------------------------------------------------------
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

# --- Chunking parameters --------------------------------------------------
# Smaller chunks → more precise retrieval; larger chunks → more context.
Settings.chunk_size = 512       # tokens per chunk
Settings.chunk_overlap = 50     # overlapping tokens between consecutive chunks

print(
    f"Settings configured:\n"
    f"  LLM          : {Settings.llm.model}\n"
    f"  Embed model  : {Settings.embed_model.model_name}\n"
    f"  Chunk size   : {Settings.chunk_size} tokens\n"
    f"  Chunk overlap: {Settings.chunk_overlap} tokens"
)

Settings configured:
  LLM          : gpt-3.5-turbo
  Embed model  : text-embedding-ada-002
  Chunk size   : 512 tokens
  Chunk overlap: 50 tokens


## 4. Load documents and build the index

In [5]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader("data").load_data()
print(f"Loaded {len(documents)} document(s)")

# The index will use the chunk_size and chunk_overlap from Settings automatically.
index = VectorStoreIndex.from_documents(documents)
print("Index built with custom chunking settings!")

Loaded 1 document(s)
Index built with custom chunking settings!


## 5. Baseline query (no reranker)

First, let's see what we get with the standard top-`k` retriever.

In [6]:
query = "What did Paul Graham work on at Viaweb and what did he learn from it?"

# Retrieve the top 6 chunks, then synthesise
baseline_engine = index.as_query_engine(similarity_top_k=6)
baseline_response = baseline_engine.query(query)

print("=== Baseline response ===")
print(baseline_response)

=== Baseline response ===
Paul Graham worked on building an online store builder software at Viaweb. He learned the importance of high production values in making users look legitimate on online stores. Additionally, he learned about retail details such as the significance of having a close-up image of a product's collar rather than a full picture of the item.


## 6. Improved retrieval with `LLMRerank`

`LLMRerank` is a **post-retrieval** step that asks the LLM to score and re-order the initial set of candidate chunks before passing them to the synthesis step. This typically improves answer quality when the initial retrieval is noisy.

**Workflow:**
1. Retrieve a **larger** candidate set (e.g. top 10) from the vector store.
2. `LLMRerank` scores each chunk against the query and keeps only the top `choice_top_n`.
3. The filtered chunks are passed to the LLM for final answer synthesis.

In [7]:
from llama_index.core.postprocessor import LLMRerank

# Reranker: ask the LLM to pick the best 3 chunks out of the top 10 retrieved.
reranker = LLMRerank(
    choice_batch_size=5,   # chunks evaluated per LLM call
    top_n=3,               # number of chunks to keep after reranking
)

rerank_engine = index.as_query_engine(
    similarity_top_k=10,               # retrieve a wider candidate set
    node_postprocessors=[reranker],    # apply the reranker before synthesis
)

rerank_response = rerank_engine.query(query)

print("=== Response with LLMRerank ===")
print(rerank_response)

=== Response with LLMRerank ===
Paul Graham worked on developing an application builder at Viaweb. He also worked on network infrastructure and the first two services - images and phone calls. From his experience at Viaweb, he learned about the importance of keeping software tight, the challenges of running a company, the value of simplicity and affordability in pricing, and the necessity of doing things that may seem desperate to attract users, such as building stores for them.


## 7. Compare the source nodes

Inspect which chunks were used in each case.

In [8]:
print("--- Baseline source nodes ---")
for i, node in enumerate(baseline_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

print()
print("--- Reranked source nodes ---")
for i, node in enumerate(rerank_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

--- Baseline source nodes ---
[1] score=0.8627 | I got so excited about this idea that I couldn't think about anything
else. It seemed obvious that this was the future. ...
[2] score=0.8515 | So we didn't reach breakeven until about
when Yahoo bought us in the summer of 1998. Which in turn meant we
were at the ...
[3] score=0.8484 | Now that I didn't
need money anymore, why was I doing this? If this vision had to be
realized as a company, then screw t...
[4] score=0.8479 | After I
moved to New York I became her de facto studio assistant.  She liked to paint on big, square canvases, 4 to 5 fe...
[5] score=0.8432 | Now we felt like we were really onto something. I had visions of a
whole new generation of software working this way. Yo...
[6] score=0.8414 | I kept the code tight
and didn't have to integrate with any other software except Robert's
and Trevor's, so it was quite...

--- Reranked source nodes ---
[1] score=10.0000 | I got so excited about this idea that I couldn't think about 

## 8. Experiment – try different settings

Use the cell below to experiment with different `chunk_size`, `chunk_overlap`, and `top_n` values to see how they affect the quality of the answer.

In [9]:
# --- Experiment with your own parameters ---
EXP_CHUNK_SIZE    = 256   # try 128, 256, 512, 1024
EXP_CHUNK_OVERLAP = 25    # try 0, 25, 50, 100
EXP_TOP_N         = 3     # number of chunks kept after reranking

Settings.chunk_size    = EXP_CHUNK_SIZE
Settings.chunk_overlap = EXP_CHUNK_OVERLAP

exp_documents = SimpleDirectoryReader("data").load_data()
exp_index     = VectorStoreIndex.from_documents(exp_documents)
exp_reranker  = LLMRerank(choice_batch_size=5, top_n=EXP_TOP_N)
exp_engine    = exp_index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[exp_reranker],
)

exp_response = exp_engine.query(query)
print(f"chunk_size={EXP_CHUNK_SIZE}, chunk_overlap={EXP_CHUNK_OVERLAP}, top_n={EXP_TOP_N}")
print(exp_response)

chunk_size=256, chunk_overlap=25, top_n=3
Paul Graham worked on an application builder at Viaweb. From his experience at Viaweb, he learned that it's better for technology companies to be run by product people rather than sales people, the importance of avoiding bugs caused by too many people editing code, the significance of office environment on productivity, the value of corridor conversations over planned meetings, the risks associated with big bureaucratic customers, and the importance of being the "entry level" option in a market to avoid being surpassed by competitors.
